# 3. fasea — Scoring API-aren demo (FastAPI)

Koaderno honek produkzio-API beraren portaera erakusten du prozesu
barruan, zerbitzurik altxatu gabe: alerta-atalaseak, sarreren
baliozkotze zorrotza, erroreen kudeaketa eta karga bakarra abioan.
Ez du APIaren kopia bat; `scoring` paketeko kode bera inportatzen du.

Bi bide daude: bundle-rik gabeko portaera (itxi segurua, 503) beti
exekutatzen da; benetako `/predict` bat baldintzazkoa da eta bundle
lokala (`outputs/api/manifest.json`) badago bakarrik egiten da.


## 1. Ingurunea eta erroa

Erroa biltegitik edo `notebooks/`-tik ebazten da. Ez dago instalaziorik
ziegetan.


In [1]:
import sys
from pathlib import Path

ROOT = (
    Path.cwd()
    if (Path.cwd() / "data/ai4i2020_train.csv").is_file()
    else Path.cwd().parent
)
sys.path.insert(0, str(ROOT))
print("ROOT:", ROOT)

from scoring.alert_policy import alert_level
from scoring.score_api import Telemetry, create_app

print("scoring inportatuta: OK")


ROOT: /home/tears/bigdata/01_Erronka1_CNC_Guard/B-taldea


scoring inportatuta: OK


## 2. Alerta-atalaseak: OK / AVISO / CRITICA

Kontratu irakaslea: <0,15 OK; >=0,15 AVISO; >=0,7 CRITICA.
`scoring.alert_policy.alert_level` funtzio berak erabakitzen du APIan
eta hemen. Mugak probatzen dira, eta balio baliogabeek salbuespena
jaurtitzen dute (ez dute probabilitaterik asmatzen).


In [2]:
for probability in (0.0, 0.1499, 0.15, 0.5, 0.6999, 0.7, 1.0):
    print(f"{probability:.4f} -> {alert_level(probability)}")

for bad in (-0.1, 1.5, float("nan")):
    try:
        alert_level(bad)
    except ValueError as exc:
        print(f"{bad} -> ValueError: {exc}")
    else:
        raise AssertionError("balio baliogabeak ezetza jaso behar zuen")


0.0000 -> OK
0.1499 -> OK
0.1500 -> AVISO
0.5000 -> AVISO
0.6999 -> AVISO
0.7000 -> CRITICA
1.0000 -> CRITICA
-0.1 -> ValueError: Probability must be finite in [0,1]
1.5 -> ValueError: Probability must be finite in [0,1]
nan -> ValueError: Probability must be finite in [0,1]


## 3. Telemetriaren baliozkotze zorrotza

Adibide balioduna `contracts/examples/valid/telemetry.json`-tik dator.
Ondoren lau sarrera baliogabe: etiketa gehigarri bat (`failure_TWF`),
balio negatibo bat (`rpm`), `event_id` labur bat eta denbora-zigilu
oker bat. Pydantic-ek denak ezeztatzen ditu; APIak 422 itzultzen du
sarrera oihukatu gabe.


In [3]:
import json

payload = json.loads((ROOT / "contracts/examples/valid/telemetry.json").read_text())
telemetry = Telemetry.model_validate(payload)
print("balioduna: OK,", telemetry.machine_id, telemetry.machine_type)

cases = {
    "etiketa gehigarria": {**payload, "failure_TWF": 1},
    "rpm negatiboa": {**payload, "rpm": -1},
    "event_id laburra": {**payload, "event_id": "abc"},
    "denbora-zigilu okerra": {**payload, "timestamp": "2026-10-15 08:09:06"},
}
for name, case in cases.items():
    try:
        Telemetry.model_validate(case)
    except Exception as exc:
        first = exc.errors()[0]
        print(f"{name} -> {type(exc).__name__}: {first['loc']} {first['msg']}")
    else:
        raise AssertionError(f"{name}: ezetza jaso behar zuen")


balioduna: OK, CNC-049 L
etiketa gehigarria -> ValidationError: ('failure_TWF',) Extra inputs are not permitted
rpm negatiboa -> ValidationError: ('rpm',) Input should be greater than or equal to 0
event_id laburra -> ValidationError: ('event_id',) String should match pattern '^[0-9a-f]{64}$'
denbora-zigilu okerra -> ValidationError: ('timestamp',) String should match pattern '^\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}(?:\.\d{1,6})?Z$'


## 4. Bundle-rik gabe: itxi segurua (fail closed)

Manifestu huts batekin, abioan karga egiten du huts eta APIak prest
ez dagoela adierazten du: `/health` eta `/predict` 503 dira, eta ez
da probabilitaterik igortzen. Gorputz erraldoi batek 413 jasotzen du.
Bezeroa `tests/api_client.py`-ren patroi bera da (lifespan esplizitua,
HTTPX ASGI garraioa), hemen lerrokatuta.


In [4]:
import tempfile
from contextlib import AsyncExitStack, asynccontextmanager

from httpx import ASGITransport, AsyncClient


@asynccontextmanager
async def api_client(app):
    lifespan = AsyncExitStack()
    try:
        await lifespan.enter_async_context(app.router.lifespan_context(app))
        async with AsyncClient(
            transport=ASGITransport(app=app), base_url="http://testserver"
        ) as client:
            yield client
    finally:
        await lifespan.aclose()


with tempfile.TemporaryDirectory() as tmp:
    manifest = Path(tmp) / "manifest.json"
    manifest.write_text("{}")
    app = create_app(manifest)
    async with api_client(app) as client:
        health = await client.get("/health")
        print("GET /health (bundle gabe):", health.status_code)
        prediction = await client.post("/predict", json=payload)
        print("POST /predict (bundle gabe):", prediction.status_code)
        huge = await client.post("/predict", content=b"x" * 33000)
        print("POST /predict (33KB, bundle gabe):", huge.status_code)
        assert (health.status_code, prediction.status_code, huge.status_code) == (503, 503, 413)
print("OK: itxi segurua eta muga bete dira.")


Model bundle unavailable or invalid; scoring remains unready
Traceback (most recent call last):
  File "/home/tears/bigdata/01_Erronka1_CNC_Guard/B-taldea/scoring/score_api.py", line 107, in lifespan
    validate_bundle_scope(manifest)
RuntimeError: Model initialization failed; details redacted


GET /health (bundle gabe): 503
POST /predict (bundle gabe): 503
POST /predict (33KB, bundle gabe): 413
OK: itxi segurua eta muga bete dira.


## 5. Bundle lokala badago: benetako `/predict` (baldintzazkoa)

Ziegi hau `outputs/api/manifest.json` existitzen bada bakarrik egiten
du benetako inferentzia; bestela, saltatu egiten da, hutsegiterik gabe.
Bundle-a dagoenean, ereduak behin kargatzen direla egiaztatzen da
(`app.state.models` objektu bera eskaeraren aurretik eta ondoren).


In [5]:
manifest = ROOT / "outputs" / "api" / "manifest.json"
if not manifest.exists():
    print("SKIP: ez dago bundle lokalik (outputs/api/manifest.json).")
    print("Bundle-a sortzeko: scripts.experiment_rf + experiment_anomaly + package_model.")
else:
    app = create_app(manifest)
    async with api_client(app) as client:
        health = await client.get("/health")
        print("GET /health:", health.status_code, health.json())
        loaded = app.state.models
        response = await client.post("/predict", json=payload)
        body = response.json()
        print("POST /predict:", response.status_code)
        print("alert_level:", body["alert_level"], "| failure_score:", round(body["failure_score"], 4))
        print("karga bakarra abioan:", app.state.models is loaded)
        assert health.status_code == 200 and response.status_code == 200
        assert body["event_id"] == payload["event_id"]
        assert app.state.models is loaded


SKIP: ez dago bundle lokalik (outputs/api/manifest.json).
Bundle-a sortzeko: scripts.experiment_rf + experiment_anomaly + package_model.


## 6. Muga zintzoak eta birsorkuntza

- Demo hau prozesu barruan doa (ez dago sareko ataka edo Docker-rik);
  ekoizpeneko latentzia/karga ez da hemen neurtzen (CG23 eta #81 dira
  horretarako bidea).
- 422/503/413 erroreak benetakoak dira, ez simulatuak; baina manipulazio
  proba osoa (`test_api.py`) test-sortan dago, ez ziega honetan.
- Alerta-atalaseak kontratu irakaslekoak dira (0,15/0,7); offline
  tuning-a `scripts.report_thresholds` eta CG10 ebidentzian dago.

Birsorkuntza (tresna efimeroekin, lock-a aldatu gabe):

```bash
uv run --frozen --with ipykernel --with nbconvert jupyter nbconvert \
  --to notebook --execute notebooks/03_scoring_api_demo.ipynb \
  --ExecutePreprocessor.kernel_name=python3 \
  --output 03_scoring_api_demo.ipynb --output-dir outputs/notebooks-exec
```
